In [1]:
import requests
import pandas as pd
import numpy as np
from keys import *

# Realiza una solicitud a la API de OpenWeather para obtener el pronóstico meteorológico
city = "Lima"
country = "PE"
response = requests.get(f'http://api.openweathermap.org/data/2.5/forecast/?q={city},{country}&appid={OWM_key}&units=metric&lang=en')
data = response.json()

In [2]:
# Extract forecast list
forecast_list = data.get('list', [])

# Prepare lists for DataFrame columns
times = []
temperatures = []
humidities = []
weather_statuses = []
wind_speeds = []
rain_volumes = []
snow_volumes = []

for entry in forecast_list:
    times.append(entry.get('dt_txt', np.nan))
    temperatures.append(entry.get('main', {}).get('temp', np.nan))
    humidities.append(entry.get('main', {}).get('humidity', np.nan))
    weather_statuses.append(entry.get('weather', [{}])[0].get('main', np.nan))
    wind_speeds.append(entry.get('wind', {}).get('speed', np.nan))
    rain_volumes.append(entry.get('rain', {}).get('3h', np.nan))
    snow_volumes.append(entry.get('snow', {}).get('3h', np.nan))

# Create DataFrame
df = pd.DataFrame({
    'weather_datetime': times,
    'temperature': temperatures,
    'humidity': humidities,
    'weather_status': weather_statuses,
    'wind': wind_speeds,
    'rain_qty': rain_volumes,
    'snow': snow_volumes,
    'municipality_iso_country': f"Lima,PE"
})

print(df.head())

      weather_datetime  temperature  humidity weather_status  wind  rain_qty  \
0  2025-10-30 00:00:00        22.17        69         Clouds  3.24       NaN   
1  2025-10-30 03:00:00        20.87        73         Clouds  3.11       NaN   
2  2025-10-30 06:00:00        19.17        79         Clouds  1.66       NaN   
3  2025-10-30 09:00:00        17.27        85         Clouds  2.68       NaN   
4  2025-10-30 12:00:00        17.00        86         Clouds  2.66       NaN   

   snow municipality_iso_country  
0   NaN                  Lima,PE  
1   NaN                  Lima,PE  
2   NaN                  Lima,PE  
3   NaN                  Lima,PE  
4   NaN                  Lima,PE  


In [3]:
import sqlalchemy
import pymysql
# connection details for the local mysql database
schema = "gans"
host = "127.0.0.1"
user = "root"
password = "cebolla2018"
port = 3306
con = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [4]:
# send the weather data to the database
df.to_sql('weather_data', if_exists = 'append', con = con, index=False)


40